# Connect4 AlphaZero scaffold, project-integrated

This notebook is a starting point for an AlphaZero-style Connect4 run using the current project code:

- `C4.CNet192.CNet192` for the policy/value network
- `C4.fast_connect4_lookahead.Connect4Lookahead` for benchmark opponents
- `PPO.ppo_agent_eval` for the existing empty-board GS evaluation and Excel logging
- optional opening-prefix evaluation for a GS-open-like signal

Modes:

- `scratch`: start from a fresh `CNet192`
- `checkpoint`: warm-start from a compatible PPO/supervised CNet192 checkpoint, for example `PPO_Models/PPO_1004.pt`
- `resume`: continue an AlphaZero checkpoint from `Models/AlphaZero/...`

Important: AlphaZero training uses terminal values in `[-1, 0, +1]`, so the value head is trained with `tanh(value_raw)`. Existing PPO/supervised checkpoints are useful warm starts because the policy head is already strong. The value head may or may not be meaningful, but MCTS will start correcting it immediately. A wise neural net admits what it doesn't know, a rare trait in software and meetings.

In [ ]:
import time
begin_start_time = time.time()
time_str = time.strftime('%Y-%m-%d %H-%M-%S', time.localtime(begin_start_time))
print(time_str)

In [ ]:
# ============================================================
# 0. Configuration
# ============================================================

from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import os
import sys
import json
import math
import time
import random
import re

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from IPython.display import display, HTML

@dataclass
class AZConfig:
    # Project/imports
    project_root: str = "."        # run notebook from project root, or set this explicitly
    run_name = "AZ_008"
    seed: int = 666
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Start mode
    mode: str = "checkpoint"       # "scratch", "checkpoint", "resume"
    init_checkpoint: str = "PPO_Models/PPO_2002.pt"
    resume_checkpoint = r"Models/AlphaZero/AZ_003.pt"
    strict_checkpoint_load: bool = False

    # Teacher anchor
    use_teacher_anchor: bool = True
    #teacher_checkpoint: str = r"Models/AlphaZero/AZ_003.pt"
    teacher_checkpoint: str = "PPO_Models/PPO_2002.pt"
    mcts_policy_mix: float = 0.20          # 0.50 = 50% MCTS, 50% teacher

    # Network
    in_channels: int = 1
    use_mid_3x3: bool = True

    # MCTS
    mcts_sims: int = 96 #64
    cpuct: float = 1.75 #1.75
    root_dirichlet_alpha: float = 0.30
    root_exploration_fraction: float = 0.20
    selfplay_temperature: float = 0.75
    selfplay_temperature_final: float = 0.10
    temperature_cutoff_ply: int = 10

    # Self-play / training loop
    iterations_to_run: int = 32 #10
    self_play_games_per_iter: int = 16 #8
    max_game_plies: int = 42
    replay_max_size: int = 250_000
    min_train_samples: int = 8192
    train_epochs_per_iter: int = 1
    train_batches_per_iter: int = 16 #200
    batch_size: int = 256
    lr: float = 1.0e-5
    weight_decay: float = 1.0e-4
    value_loss_weight: float = 0.03
    grad_clip_norm: float = 1.0

    # Evaluation / checkpointing
    eval_every: int = 2
    eval_full_suite: bool = False       # False = EVAL_CFG, True = EVALUATION_OPPONENTS
    eval_empty_board: bool = True       # true EVAL_CFG
    eval_openings: bool = False
    eval_excel_path: str = "Logs/AlphaZero/AZ_eval_empty.xlsx"
    eval_open_excel_path: str = "Logs/AlphaZero/AZ_eval_openings.xlsx"
    save_every: int = 1
    save_replay: bool = True
    plot_every: int = 1


CFG = AZConfig()

PROJECT_ROOT = Path(CFG.project_root).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_DIR = PROJECT_ROOT / "Models" / "AlphaZero" / CFG.run_name
LOG_DIR = PROJECT_ROOT / "Logs" / "AlphaZero"
PLOT_DIR = PROJECT_ROOT / "Plots" / "AlphaZero" / CFG.run_name
for p in [RUN_DIR, LOG_DIR, PLOT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

DEVICE = torch.device(CFG.device)
print("DEVICE:", DEVICE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DIR:", RUN_DIR)

In [ ]:
# ============================================================
# 1. Project imports
# ============================================================

from C4.CNet192 import CNet192, save_cnet192, load_cnet192
from C4.fast_connect4_lookahead import Connect4Lookahead
from C4.eval_oppo_dict import EVAL_CFG, EVALUATION_OPPONENTS, OPENING_NOISE_K

from PPO.ppo_agent_eval import (
    evaluate_suite,
    evaluate_and_log_to_excel,
    global_score_from_suite_df,
    make_opponent,
    round_robin_matrix,
    metascores_from_matrix,
    plot_rr_heatmap,
)

try:
    from PPO.ppo_live_plot import plot_live_training_ppo
    HAS_PPO_LIVE_PLOT = True
except Exception as e:
    HAS_PPO_LIVE_PLOT = False
    print("PPO live plot import skipped:", repr(e))

LA_SHARED = Connect4Lookahead()
LA_SHARED.OPENING_RANDOM = False

ROWS, COLS = 6, 7
CENTER_ORDER = [3, 4, 2, 5, 1, 6, 0]
NEG_INF = -1.0e9

print("OK: CNet192, Connect4Lookahead, eval helpers imported")

In [ ]:
# ============================================================
# 2. Board utilities, top-row index 0, values {-1, 0, +1}
# ============================================================

def empty_board() -> np.ndarray:
    return np.zeros((ROWS, COLS), dtype=np.int8)


def legal_actions(board: np.ndarray) -> List[int]:
    return [c for c in range(COLS) if int(board[0, c]) == 0]


def other_player(player: int) -> int:
    return -int(player)


def apply_action(board: np.ndarray, col: int, player: int) -> np.ndarray:
    col = int(col)
    if col < 0 or col >= COLS or board[0, col] != 0:
        raise ValueError(f"Illegal move {col}")
    out = board.copy()
    for r in range(ROWS - 1, -1, -1):
        if out[r, col] == 0:
            out[r, col] = np.int8(player)
            return out
    raise ValueError(f"Column {col} is full")


def has_winner(board: np.ndarray, player: int) -> bool:
    p = int(player)
    # horizontal
    for r in range(ROWS):
        for c in range(COLS - 3):
            if np.all(board[r, c:c+4] == p):
                return True
    # vertical
    for r in range(ROWS - 3):
        for c in range(COLS):
            if np.all(board[r:r+4, c] == p):
                return True
    # diagonals
    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            if all(board[r+i, c+i] == p for i in range(4)):
                return True
            if all(board[r+3-i, c+i] == p for i in range(4)):
                return True
    return False


def terminal_status(board: np.ndarray) -> Tuple[bool, int]:
    """Return (done, winner), winner in {+1, -1, 0}."""
    if has_winner(board, 1):
        return True, 1
    if has_winner(board, -1):
        return True, -1
    if len(legal_actions(board)) == 0:
        return True, 0
    return False, 0


def board_to_pov_state(board: np.ndarray, player: int) -> np.ndarray:
    """Return current-player POV state with shape (1, 6, 7)."""
    return (board.astype(np.float32) * float(player))[None, :, :]


def mirror_state_policy(state: np.ndarray, pi: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Mirror columns for data augmentation."""
    return state[:, :, ::-1].copy(), np.asarray(pi, dtype=np.float32)[::-1].copy()


def mask_logits_by_state(logits: torch.Tensor, states: torch.Tensor) -> torch.Tensor:
    """
    Mask illegal columns using the top row of POV states.
    states: (B,1,6,7), logits: (B,7)
    """
    illegal = states[:, 0, 0, :] != 0
    return logits.masked_fill(illegal, NEG_INF)


def softmax_legal_from_logits(logits: torch.Tensor, board: np.ndarray) -> np.ndarray:
    logits = logits.detach().float().clone()
    for c in range(COLS):
        if board[0, c] != 0:
            logits[c] = NEG_INF
    probs = torch.softmax(logits, dim=-1).cpu().numpy().astype(np.float64)
    probs = np.nan_to_num(probs, nan=0.0, posinf=0.0, neginf=0.0)
    s = float(probs.sum())
    if s <= 0.0:
        legal = legal_actions(board)
        probs[:] = 0.0
        if legal:
            probs[legal] = 1.0 / len(legal)
    else:
        probs /= s
    return probs


def pick_action_from_pi(pi: np.ndarray, board: np.ndarray, rng: np.random.Generator) -> int:
    legal = legal_actions(board)
    if not legal:
        return -1
    p = np.asarray(pi, dtype=np.float64).copy()
    illegal = [c for c in range(COLS) if c not in legal]
    p[illegal] = 0.0
    s = float(p.sum())
    if s <= 0.0 or not np.isfinite(s):
        return int(rng.choice(legal))
    p /= s
    return int(rng.choice(np.arange(COLS), p=p))

print("Board utilities ready")

In [ ]:
# ============================================================
# 3. Model + checkpoint utilities
# ============================================================

def make_fresh_model(cfg: AZConfig) -> CNet192:
    return CNet192(in_channels=int(cfg.in_channels), use_mid_3x3=bool(cfg.use_mid_3x3)).to(DEVICE)


def _extract_state_dict(payload: Any) -> Dict[str, torch.Tensor]:
    """Accept save_cnet192, plain state_dict, or common nested checkpoint shapes."""
    if isinstance(payload, dict):
        # Plain state_dict: most values are tensors.
        if payload and all(isinstance(v, torch.Tensor) for v in payload.values()):
            return payload
        for key in ["model_state_dict", "state_dict", "net_state_dict", "policy_state_dict", "actor_state_dict"]:
            if key in payload and isinstance(payload[key], dict):
                return payload[key]
        for key in ["model", "net", "policy", "actor_critic"]:
            obj = payload.get(key, None)
            if isinstance(obj, dict):
                return obj
            if hasattr(obj, "state_dict"):
                return obj.state_dict()
    raise KeyError("Could not find a model state_dict in checkpoint payload")


def _strip_prefix_if_present(sd: Dict[str, torch.Tensor], prefix: str) -> Dict[str, torch.Tensor]:
    if not any(k.startswith(prefix) for k in sd.keys()):
        return sd
    return {k[len(prefix):] if k.startswith(prefix) else k: v for k, v in sd.items()}


def load_flexible_cnet192_checkpoint(path: str | Path, cfg: AZConfig) -> Tuple[CNet192, Dict[str, Any]]:
    """
    Load a CNet192-ish checkpoint into CNet192.
    First tries project load_cnet192(), then falls back to flexible key extraction.
    """
    path = Path(path)
    try:
        model, ckpt = load_cnet192(path, device=DEVICE, strict=bool(cfg.strict_checkpoint_load))
        model.to(DEVICE)
        print(f"Loaded via load_cnet192: {path}")
        return model, ckpt
    except Exception as e:
        print("load_cnet192 failed, trying flexible loader:", repr(e))

    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    sd = _extract_state_dict(ckpt)
    for pref in ["module.", "model.", "net.", "policy.", "actor_critic."]:
        sd = _strip_prefix_if_present(sd, pref)

    model = make_fresh_model(cfg)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f"Loaded flexible checkpoint: {path}")
    print("missing keys:", missing)
    print("unexpected keys:", unexpected)
    return model, ckpt if isinstance(ckpt, dict) else {"raw_payload_type": str(type(ckpt))}


@torch.no_grad()
def policy_value_for_board(model: nn.Module, board: np.ndarray, player: int) -> Tuple[np.ndarray, float]:
    """Return legal policy probabilities and tanh(value), both from current player POV."""
    model.eval()
    state = board_to_pov_state(board, player)
    x = torch.from_numpy(state[None, :, :, :]).to(DEVICE)  # (1,1,6,7)
    out = model(x)
    if isinstance(out, (tuple, list)):
        logits, value_raw = out[0], out[1]
        value = float(torch.tanh(value_raw.reshape(-1)[0]).detach().cpu())
    else:
        logits = out
        value = 0.0
    probs = softmax_legal_from_logits(logits[0], board)
    return probs, value


@torch.no_grad()
def greedy_model_action(model: nn.Module, board: np.ndarray, player: int) -> int:
    pi, _ = policy_value_for_board(model, board, player)
    legal = legal_actions(board)
    if not legal:
        return -1
    best = max(float(pi[c]) for c in legal)
    tied = [c for c in legal if abs(float(pi[c]) - best) <= 1e-12]
    return int(sorted(tied, key=lambda c: (abs(c - 3), c))[0])

print("Model utilities ready")

In [ ]:
@torch.no_grad()
def policy_probs_from_states(model: nn.Module, states: torch.Tensor) -> torch.Tensor:
    """
    Frozen teacher policy on already-POV states.

    states: (B,1,6,7), current-player POV.
    returns: (B,7), legal-masked normalized probabilities.
    """
    was_training = bool(getattr(model, "training", False))
    model.eval()

    out = model(states)
    logits = out[0] if isinstance(out, (tuple, list)) else out
    logits = mask_logits_by_state(logits, states)

    probs = torch.softmax(logits, dim=-1)
    probs = torch.nan_to_num(probs, nan=0.0, posinf=0.0, neginf=0.0)

    # Explicit illegal zeroing, then renormalize.
    illegal = states[:, 0, 0, :] != 0
    probs = probs.masked_fill(illegal, 0.0)

    s = probs.sum(dim=-1, keepdim=True)
    legal_count = (~illegal).sum(dim=-1, keepdim=True).clamp_min(1)

    uniform = (~illegal).float() / legal_count.float()
    probs = torch.where(s > 1.0e-12, probs / s.clamp_min(1.0e-12), uniform)

    if was_training:
        model.train(True)

    return probs


def normalize_policy_targets(pi: torch.Tensor, states: torch.Tensor) -> torch.Tensor:
    """
    Keep policy targets legal and normalized.
    Useful after mixing MCTS and teacher policies.
    """
    pi = torch.nan_to_num(pi.float(), nan=0.0, posinf=0.0, neginf=0.0)

    illegal = states[:, 0, 0, :] != 0
    pi = pi.masked_fill(illegal, 0.0)

    s = pi.sum(dim=-1, keepdim=True)
    legal_count = (~illegal).sum(dim=-1, keepdim=True).clamp_min(1)

    uniform = (~illegal).float() / legal_count.float()
    pi = torch.where(s > 1.0e-12, pi / s.clamp_min(1.0e-12), uniform)

    return pi

In [ ]:
# ============================================================
# 4. PUCT MCTS
# ============================================================

class MCTSEdge:
    __slots__ = ("prior", "N", "W", "child")
    def __init__(self, prior: float):
        self.prior = float(prior)
        self.N = 0
        self.W = 0.0
        self.child: Optional[MCTSNode] = None

    @property
    def Q(self) -> float:
        return self.W / self.N if self.N > 0 else 0.0


class MCTSNode:
    __slots__ = ("board", "player", "children")
    def __init__(self, board: np.ndarray, player: int):
        self.board = board
        self.player = int(player)
        self.children: Dict[int, MCTSEdge] = {}

    @property
    def expanded(self) -> bool:
        return len(self.children) > 0

    def expand(self, priors: np.ndarray) -> None:
        legal = legal_actions(self.board)
        if not legal:
            return
        p = np.asarray(priors, dtype=np.float64).copy()
        p[[c for c in range(COLS) if c not in legal]] = 0.0
        s = float(p.sum())
        if s <= 0.0 or not np.isfinite(s):
            p[:] = 0.0
            p[legal] = 1.0 / len(legal)
        else:
            p /= s
        self.children = {int(a): MCTSEdge(float(p[a])) for a in legal}

    def add_root_noise(self, alpha: float, frac: float, rng: np.random.Generator) -> None:
        if not self.children or frac <= 0.0:
            return
        actions = list(self.children.keys())
        noise = rng.dirichlet([float(alpha)] * len(actions))
        for a, n in zip(actions, noise):
            e = self.children[a]
            e.prior = (1.0 - float(frac)) * e.prior + float(frac) * float(n)

    def select(self, cpuct: float) -> Tuple[int, MCTSEdge]:
        total_n = sum(e.N for e in self.children.values())
        sqrt_n = math.sqrt(max(1, total_n))
        best_a = None
        best_e = None
        best_score = -1.0e100
        for a, e in self.children.items():
            u = float(cpuct) * e.prior * sqrt_n / (1 + e.N)
            score = e.Q + u
            if score > best_score:
                best_score = score
                best_a = a
                best_e = e
        return int(best_a), best_e


def terminal_value_from_player_pov(board: np.ndarray, player: int) -> Tuple[bool, float]:
    done, winner = terminal_status(board)
    if not done:
        return False, 0.0
    if winner == 0:
        return True, 0.0
    return True, 1.0 if int(winner) == int(player) else -1.0


def mcts_simulation(root: MCTSNode, model: nn.Module, cfg: AZConfig) -> None:
    node = root
    path: List[MCTSEdge] = []

    while True:
        done, value = terminal_value_from_player_pov(node.board, node.player)
        if done:
            break

        if not node.expanded:
            priors, value = policy_value_for_board(model, node.board, node.player)
            node.expand(priors)
            break

        action, edge = node.select(cfg.cpuct)
        if edge.child is None:
            next_board = apply_action(node.board, action, node.player)
            edge.child = MCTSNode(next_board, other_player(node.player))
        path.append(edge)
        node = edge.child

    # value is from current leaf node.player POV.
    # Each edge stores value from its parent node POV, so flip sign per ply.
    for edge in reversed(path):
        value = -value
        edge.N += 1
        edge.W += float(value)


def visits_to_policy(root: MCTSNode, temperature: float) -> np.ndarray:
    pi = np.zeros(COLS, dtype=np.float64)
    if not root.children:
        legal = legal_actions(root.board)
        if legal:
            pi[legal] = 1.0 / len(legal)
        return pi.astype(np.float32)

    actions = list(root.children.keys())
    visits = np.array([root.children[a].N for a in actions], dtype=np.float64)

    if float(temperature) <= 1.0e-6:
        best_n = np.max(visits)
        best_actions = [a for a, n in zip(actions, visits) if n == best_n]
        # center tie-break, because Connect4 has taste
        a = sorted(best_actions, key=lambda c: (abs(c - 3), c))[0]
        pi[a] = 1.0
        return pi.astype(np.float32)

    visits = np.power(visits, 1.0 / float(temperature))
    s = float(visits.sum())
    if s <= 0.0 or not np.isfinite(s):
        for a in actions:
            pi[a] = 1.0 / len(actions)
    else:
        for a, v in zip(actions, visits / s):
            pi[a] = float(v)
    return pi.astype(np.float32)


def run_mcts(
    model: nn.Module,
    board: np.ndarray,
    player: int,
    cfg: AZConfig,
    rng: np.random.Generator,
    add_root_noise: bool = True,
    visit_temperature: float = 1.0,
) -> np.ndarray:
    root = MCTSNode(board.copy(), int(player))
    done, _ = terminal_value_from_player_pov(root.board, root.player)
    if done:
        return visits_to_policy(root, temperature=1.0)

    priors, _ = policy_value_for_board(model, root.board, root.player)
    root.expand(priors)
    if add_root_noise:
        root.add_root_noise(cfg.root_dirichlet_alpha, cfg.root_exploration_fraction, rng)

    for _ in range(int(cfg.mcts_sims)):
        mcts_simulation(root, model, cfg)

    return visits_to_policy(root, temperature=float(visit_temperature))

print("MCTS ready")

In [ ]:
# ============================================================
# 5. Replay buffer and dataset
# ============================================================

class AZReplayBuffer:
    def __init__(self, max_size: int):
        self.max_size = int(max_size)
        self.states: List[np.ndarray] = []
        self.policies: List[np.ndarray] = []
        self.values: List[float] = []

    def __len__(self) -> int:
        return len(self.values)

    def add(self, state: np.ndarray, pi: np.ndarray, z: float, augment_mirror: bool = True) -> None:
        self.states.append(np.asarray(state, dtype=np.float32).copy())
        self.policies.append(np.asarray(pi, dtype=np.float32).copy())
        self.values.append(float(z))

        if augment_mirror:
            st_m, pi_m = mirror_state_policy(state, pi)
            self.states.append(st_m.astype(np.float32, copy=False))
            self.policies.append(pi_m.astype(np.float32, copy=False))
            self.values.append(float(z))

        overflow = len(self.values) - self.max_size
        if overflow > 0:
            del self.states[:overflow]
            del self.policies[:overflow]
            del self.values[:overflow]

    def sample_indices(self, n: int, rng: np.random.Generator) -> np.ndarray:
        return rng.integers(0, len(self.values), size=int(n), endpoint=False)

    def save_npz(self, path: str | Path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        if len(self.values) == 0:
            np.savez_compressed(path, states=np.zeros((0,1,6,7), dtype=np.float32), policies=np.zeros((0,7), dtype=np.float32), values=np.zeros((0,), dtype=np.float32))
            return
        np.savez_compressed(
            path,
            states=np.stack(self.states).astype(np.float32),
            policies=np.stack(self.policies).astype(np.float32),
            values=np.asarray(self.values, dtype=np.float32),
        )

    @classmethod
    def load_npz(cls, path: str | Path, max_size: int) -> "AZReplayBuffer":
        path = Path(path)
        buf = cls(max_size=max_size)
        data = np.load(path)
        states = data["states"]
        policies = data["policies"]
        values = data["values"]
        start = max(0, len(values) - int(max_size))
        buf.states = [s.astype(np.float32, copy=True) for s in states[start:]]
        buf.policies = [p.astype(np.float32, copy=True) for p in policies[start:]]
        buf.values = [float(v) for v in values[start:]]
        return buf


class AZDataset(Dataset):
    def __init__(self, buffer: AZReplayBuffer, indices: np.ndarray):
        self.buffer = buffer
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self) -> int:
        return int(len(self.indices))

    def __getitem__(self, i: int):
        j = int(self.indices[i])
        return (
            torch.from_numpy(self.buffer.states[j]),
            torch.from_numpy(self.buffer.policies[j]),
            torch.tensor(self.buffer.values[j], dtype=torch.float32),
        )

print("Replay buffer ready")

In [ ]:
# ============================================================
# 6. Self-play generation
# ============================================================

def self_play_game(model: nn.Module, cfg: AZConfig, seed: int) -> Dict[str, Any]:
    rng = np.random.default_rng(int(seed))
    board = empty_board()
    player = 1
    trajectory: List[Tuple[np.ndarray, np.ndarray, int]] = []

    for ply in range(int(cfg.max_game_plies)):
        done, winner = terminal_status(board)
        if done:
            break

        temp = cfg.selfplay_temperature if ply < cfg.temperature_cutoff_ply else cfg.selfplay_temperature_final
        pi = run_mcts(
            model=model,
            board=board,
            player=player,
            cfg=cfg,
            rng=rng,
            add_root_noise=True,
            visit_temperature=float(temp),
        )

        state = board_to_pov_state(board, player)
        trajectory.append((state, pi, player))

        action = pick_action_from_pi(pi, board, rng)
        if action < 0:
            winner = 0
            break

        board = apply_action(board, action, player)
        done, winner = terminal_status(board)
        if done:
            break
        player = other_player(player)
    else:
        done, winner = terminal_status(board)
        if not done:
            winner = 0

    samples = []
    for state, pi, state_player in trajectory:
        if winner == 0:
            z = 0.0
        else:
            z = 1.0 if int(winner) == int(state_player) else -1.0
        samples.append((state, pi, z))

    return {
        "winner": int(winner),
        "plies": len(trajectory),
        "samples": samples,
        "final_board": board,
    }


def generate_self_play_batch(model: nn.Module, cfg: AZConfig, iteration: int) -> List[Dict[str, Any]]:
    games = []
    base_seed = int(cfg.seed + 100_000 * iteration)
    iterator = tqdm(range(int(cfg.self_play_games_per_iter)), desc=f"Self-play iter {iteration}", leave=True)
    for g in iterator:
        out = self_play_game(model, cfg, seed=base_seed + g)
        games.append(out)
        w = sum(1 for x in games if x["winner"] == 1)
        l = sum(1 for x in games if x["winner"] == -1)
        d = sum(1 for x in games if x["winner"] == 0)
        iterator.set_postfix(W1=w, W2=l, D=d, avg_plies=f"{np.mean([x['plies'] for x in games]):.1f}")
    return games

print("Self-play ready")

In [ ]:
# ============================================================
# 7. Training step
# ============================================================

def train_one_iteration(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    buffer: AZReplayBuffer,
    cfg: AZConfig,
    iteration: int,
    teacher_model: Optional[nn.Module] = None,
) -> Dict[str, float]:

    if len(buffer) < int(cfg.min_train_samples):
        print(f"Replay too small: {len(buffer):,} / {cfg.min_train_samples:,}. Skipping train.")
        return {
            "loss": float("nan"),
            "policy_loss": float("nan"),
            "value_loss": float("nan"),
            "entropy": float("nan"),
            "value_mse": float("nan"),
            "teacher_ce": float("nan"),
            "mcts_ce": float("nan"),
            "mcts_mix": float(getattr(cfg, "mcts_policy_mix", 1.0)),
        }

    model.train()
    if teacher_model is not None:
        teacher_model.eval()

    rng = np.random.default_rng(int(cfg.seed + 900_000 + iteration))

    losses = []
    pol_losses = []
    val_losses = []
    entropies = []
    value_mses = []
    teacher_ces = []
    mcts_ces = []

    use_teacher = bool(getattr(cfg, "use_teacher_anchor", False)) and teacher_model is not None
    mcts_mix = float(getattr(cfg, "mcts_policy_mix", 1.0))
    mcts_mix = max(0.0, min(1.0, mcts_mix))
    teacher_mix = 1.0 - mcts_mix

    total_batches = int(cfg.train_batches_per_iter) * int(cfg.train_epochs_per_iter)
    pbar = tqdm(range(total_batches), desc=f"Train iter {iteration}", leave=True)

    for _ in pbar:
        idx = buffer.sample_indices(cfg.batch_size, rng)
        ds = AZDataset(buffer, idx)
        loader = DataLoader(ds, batch_size=int(cfg.batch_size), shuffle=False, num_workers=0)

        states, target_pi_mcts, target_z = next(iter(loader))
        states = states.to(DEVICE).float()              # (B,1,6,7)
        target_pi_mcts = target_pi_mcts.to(DEVICE).float()
        target_z = target_z.to(DEVICE).float()

        target_pi_mcts = normalize_policy_targets(target_pi_mcts, states)

        if use_teacher:
            with torch.no_grad():
                target_pi_teacher = policy_probs_from_states(teacher_model, states)
                target_pi = teacher_mix * target_pi_teacher + mcts_mix * target_pi_mcts
                target_pi = normalize_policy_targets(target_pi, states)
        else:
            target_pi_teacher = None
            target_pi = target_pi_mcts

        logits, value_raw = model(states)
        logits = mask_logits_by_state(logits, states)

        logp = F.log_softmax(logits, dim=-1)
        probs = torch.softmax(logits, dim=-1)

        policy_loss = -(target_pi * logp).sum(dim=-1).mean()

        value_pred = torch.tanh(value_raw.reshape(-1))
        value_loss = F.mse_loss(value_pred, target_z)

        loss = policy_loss + float(cfg.value_loss_weight) * value_loss

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        if cfg.grad_clip_norm and cfg.grad_clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg.grad_clip_norm))

        optimizer.step()

        entropy = -(probs * logp).sum(dim=-1).mean()
        value_mse = F.mse_loss(value_pred.detach(), target_z).item()

        with torch.no_grad():
            mcts_ce = -(target_pi_mcts * logp.detach()).sum(dim=-1).mean().item()
            if target_pi_teacher is not None:
                teacher_ce = -(target_pi_teacher * logp.detach()).sum(dim=-1).mean().item()
            else:
                teacher_ce = float("nan")

        losses.append(float(loss.item()))
        pol_losses.append(float(policy_loss.item()))
        val_losses.append(float(value_loss.item()))
        entropies.append(float(entropy.item()))
        value_mses.append(float(value_mse))
        teacher_ces.append(float(teacher_ce))
        mcts_ces.append(float(mcts_ce))

        pbar.set_postfix(
            loss=f"{np.mean(losses[-20:]):.4f}",
            pi=f"{np.mean(pol_losses[-20:]):.4f}",
            v=f"{np.mean(val_losses[-20:]):.4f}",
            mix=f"{mcts_mix:.2f}",
        )

    return {
        "loss": float(np.mean(losses)),
        "policy_loss": float(np.mean(pol_losses)),
        "value_loss": float(np.mean(val_losses)),
        "entropy": float(np.mean(entropies)),
        "value_mse": float(np.mean(value_mses)),
        "teacher_ce": float(np.nanmean(teacher_ces)),
        "mcts_ce": float(np.mean(mcts_ces)),
        "mcts_mix": float(mcts_mix),
    }

print("Training step ready")

In [ ]:
# ============================================================
# 8. Checkpoint save/load
# ============================================================

def make_optimizer(model: nn.Module, cfg: AZConfig) -> torch.optim.Optimizer:
    return torch.optim.AdamW(model.parameters(), lr=float(cfg.lr), weight_decay=float(cfg.weight_decay))


def save_az_checkpoint(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    buffer: AZReplayBuffer,
    history: List[Dict[str, Any]],
    benchmark_history: Dict[str, Any],
    cfg: AZConfig,
    iteration: int,
) -> Path:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    replay_path = None
    if cfg.save_replay:
        replay_path = RUN_DIR / f"{cfg.run_name}_replay_iter_{iteration:04d}.npz"
        buffer.save_npz(replay_path)

    ckpt_path = RUN_DIR / f"{cfg.run_name}_iter_{iteration:04d}.pt"
    payload = {
        "kind": "connect4_alphazero",
        "iteration": int(iteration),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "cfg": asdict(cfg),
        "history": history,
        "benchmark_history": benchmark_history,
        "replay_path": str(replay_path) if replay_path else None,
    }
    torch.save(payload, ckpt_path)

    # Also save latest pointer.
    latest_path = RUN_DIR / f"{cfg.run_name}_latest.pt"
    torch.save(payload, latest_path)
    print(f"Saved AlphaZero checkpoint: {ckpt_path}")
    return ckpt_path


def load_az_checkpoint(path: str | Path, cfg: AZConfig) -> Tuple[nn.Module, torch.optim.Optimizer, AZReplayBuffer, List[Dict[str, Any]], Dict[str, Any], int]:
    path = Path(path)
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model = make_fresh_model(cfg)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.to(DEVICE)
    optimizer = make_optimizer(model, cfg)
    if "optimizer_state_dict" in ckpt and ckpt["optimizer_state_dict"] is not None:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    replay_path = ckpt.get("replay_path")
    if replay_path and Path(replay_path).exists():
        buffer = AZReplayBuffer.load_npz(replay_path, max_size=cfg.replay_max_size)
    else:
        print("No replay file found in checkpoint; starting with empty replay.")
        buffer = AZReplayBuffer(max_size=cfg.replay_max_size)

    history = list(ckpt.get("history", []))
    benchmark_history = dict(ckpt.get("benchmark_history", {"episode": [], "global_score": [], "by_opponent": {}}))
    start_iteration = int(ckpt.get("iteration", 0))
    print(f"Resumed {path} at iteration {start_iteration}, replay={len(buffer):,}")
    return model, optimizer, buffer, history, benchmark_history, start_iteration


def export_cnet192_policy_checkpoint(model: nn.Module, path: str | Path, cfg: AZConfig, **meta: Any) -> None:
    """Save a CNet192-compatible checkpoint for your existing PPO eval / soup tooling."""
    path = Path(path)
    save_cnet192(
        path=path,
        model=model,
        cfg={
            "arch": "cnet192",
            "input_channels": int(cfg.in_channels),
            "use_mid_3x3": bool(cfg.use_mid_3x3),
            "source": "alphazero",
            "run_name": cfg.run_name,
        },
        **meta,
    )
    print(f"Exported CNet192-compatible checkpoint: {path}")

print("Checkpoint utilities ready")

In [ ]:
# ============================================================
# 9. Empty-board GS and opening-prefix GS-open-like eval hooks
# ============================================================

OPENING_PREFIXES = [
    (),
    (3,),
    (2,),
    (4,),
    (3, 2),
    (3, 4),
    (2, 3),
    (4, 3),
    (3, 2, 3),
    (3, 4, 3),
]
MIN_GAMES_PER_OPENING_SERIES = 2 * len(OPENING_PREFIXES)


def play_model_vs_opponent_with_opening(
    model: nn.Module,
    opponent_label: str,
    opening_prefix: Tuple[int, ...],
    policy_player: int,
    lookahead: Connect4Lookahead,
    seed: int,
    max_plies: int = 42,
) -> Tuple[int, int]:
    """
    Return (result, plies), result from model POV: +1 win, 0 draw, -1 loss.
    This is intentionally close to the absolute-ish opening sweep idea.
    """
    rng = np.random.default_rng(int(seed))
    opponent = make_opponent(opponent_label, lookahead=lookahead, rng=rng)

    board = empty_board()
    player = 1
    plies = 0

    # Apply fixed opening prefix.
    for a0 in opening_prefix:
        legal = legal_actions(board)
        if not legal:
            break
        a = int(a0)
        if a not in legal:
            a = int(rng.choice(legal))
        board = apply_action(board, a, player)
        plies += 1
        done, winner = terminal_status(board)
        if done:
            if winner == 0:
                return 0, plies
            return (1 if winner == policy_player else -1), plies
        player = other_player(player)

    # Continue game.
    while plies < int(max_plies):
        legal = legal_actions(board)
        if not legal:
            return 0, plies

        if player == int(policy_player):
            a = greedy_model_action(model, board, player)
        else:
            # make_opponent expects a bitboard-ish mask, but its lookahead branch ignores mask.
            # Baseline branches can still work from board via lookahead.baseline_action.
            a = int(opponent(board, player, np.uint64(0)))

        if a not in legal:
            # Illegal move loses for the mover.
            winner = other_player(player)
            return (1 if winner == policy_player else -1), plies

        board = apply_action(board, a, player)
        plies += 1
        done, winner = terminal_status(board)
        if done:
            if winner == 0:
                return 0, plies
            return (1 if winner == policy_player else -1), plies
        player = other_player(player)

    return 0, plies


def evaluate_opening_suite(
    model: nn.Module,
    opponents_cfg: Dict[str, int],
    lookahead: Connect4Lookahead,
    seed: int,
    opening_prefixes: List[Tuple[int, ...]] = OPENING_PREFIXES,
) -> pd.DataFrame:
    rows = []
    items = list(opponents_cfg.items())
    pbar = tqdm(items, desc="Opening-prefix eval", leave=True)
    for label, n0 in pbar:
        n_games = max(int(n0), MIN_GAMES_PER_OPENING_SERIES)
        wins = losses = draws = 0
        plies_sum = 0
        for i in range(n_games):
            prefix = tuple(opening_prefixes[(i // 2) % len(opening_prefixes)])
            policy_player = 1 if (i % 2 == 0) else -1
            result, plies = play_model_vs_opponent_with_opening(
                model=model,
                opponent_label=label,
                opening_prefix=prefix,
                policy_player=policy_player,
                lookahead=lookahead,
                seed=int(seed + 10_000 * (hash(label) % 1000) + i),
            )
            wins += int(result == 1)
            losses += int(result == -1)
            draws += int(result == 0)
            plies_sum += int(plies)
        rows.append({
            "opponent": label,
            "games": n_games,
            "wins": wins,
            "losses": losses,
            "draws": draws,
            "win_rate": wins / n_games,
            "loss_rate": losses / n_games,
            "draw_rate": draws / n_games,
            "score": (wins + 0.5 * draws) / n_games,
            "avg_plies": plies_sum / n_games,
        })
        pbar.set_postfix(label=label, wr=f"{wins/n_games:.3f}")
    return pd.DataFrame(rows)


def append_suite_row_to_excel(row_df: pd.DataFrame, excel_path: str | Path) -> None:
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)
    if excel_path.exists():
        old = pd.read_excel(excel_path)
        out = pd.concat([old, row_df.reset_index()], ignore_index=True)
    else:
        out = row_df.reset_index()
    out.to_excel(excel_path, index=False)


def suite_to_summary_row(run_tag: str, suite_df: pd.DataFrame, iteration: int, score_col: str = "GLOBAL_SCORE") -> pd.DataFrame:
    row = {"TRAINING_SESSION": str(run_tag), "ITERATION": int(iteration)}
    for _, r in suite_df.iterrows():
        label = str(r["opponent"])
        col = label.replace("Lookahead-", "LA-")
        row[col] = float(r["win_rate"])
    row[score_col] = float(global_score_from_suite_df(suite_df, base=1.4))
    return pd.DataFrame([row]).set_index("TRAINING_SESSION")


def evaluate_project_hooks(model: nn.Module, cfg: AZConfig, iteration: int) -> Dict[str, Any]:
    opponents = EVALUATION_OPPONENTS if cfg.eval_full_suite else EVAL_CFG
    result: Dict[str, Any] = {}

    if cfg.eval_empty_board:
        suite_df, row_df = evaluate_and_log_to_excel(
            policy=model,
            opponents_cfg=opponents,
            excel_path=str(PROJECT_ROOT / cfg.eval_excel_path),
            run_tag=f"{cfg.run_name}_iter_{iteration:04d}",
            device=DEVICE,
            lookahead=LA_SHARED,
            seed=int(cfg.seed + iteration),
            episodes=int(iteration),
            progress=True,
            global_base=1.4,
        )
        result["empty_suite_df"] = suite_df
        result["empty_row_df"] = row_df
        result["GS"] = float(row_df["GLOBAL_SCORE"].iloc[0])

    if cfg.eval_openings:
        open_df = evaluate_opening_suite(
            model=model,
            opponents_cfg=opponents,
            lookahead=LA_SHARED,
            seed=int(cfg.seed + 20_000 + iteration),
        )
        open_row = suite_to_summary_row(
            run_tag=f"{cfg.run_name}_iter_{iteration:04d}",
            suite_df=open_df,
            iteration=iteration,
            score_col="GS_OPEN",
        )
        append_suite_row_to_excel(open_row, PROJECT_ROOT / cfg.eval_open_excel_path)
        result["open_suite_df"] = open_df
        result["open_row_df"] = open_row
        result["GS_open"] = float(open_row["GS_OPEN"].iloc[0])

    if "GS" in result and "GS_open" in result:
        result["GS_AVG"] = 0.5 * (float(result["GS"]) + float(result["GS_open"]))

    return result

print("Evaluation hooks ready")

In [ ]:
# ============================================================
# 10. History helpers and live plot
# ============================================================

def init_benchmark_history() -> Dict[str, Any]:
    return {"episode": [], "global_score": [], "global_score_open": [], "by_opponent": {}, "by_opponent_open": {}}


def update_benchmark_history(benchmark_history: Dict[str, Any], iteration: int, eval_result: Dict[str, Any]) -> None:
    benchmark_history.setdefault("episode", []).append(int(iteration))

    gs = float(eval_result.get("GS", np.nan))
    gs_open = float(eval_result.get("GS_open", np.nan))
    benchmark_history.setdefault("global_score", []).append(gs)
    benchmark_history.setdefault("global_score_open", []).append(gs_open)

    if "empty_suite_df" in eval_result:
        by = benchmark_history.setdefault("by_opponent", {})
        for _, r in eval_result["empty_suite_df"].iterrows():
            by.setdefault(str(r["opponent"]), []).append(float(r["win_rate"]))

    if "open_suite_df" in eval_result:
        by_open = benchmark_history.setdefault("by_opponent_open", {})
        for _, r in eval_result["open_suite_df"].iterrows():
            by_open.setdefault(str(r["opponent"]), []).append(float(r["win_rate"]))


def plot_live_training_az(history: List[Dict[str, Any]], benchmark_history: Dict[str, Any], title: str = "AlphaZero Connect4"):
    if not history:
        print("No history yet.")
        return None

    df = pd.DataFrame(history)
    fig, axes = plt.subplots(5, 1, figsize=(16, 18), sharex=False)

    ax = axes[0]
    ax.plot(df["iteration"], df["samples"], marker="o", label="new samples")
    ax.plot(df["iteration"], df["replay_size"], marker=".", label="replay size")
    ax.set_title("Replay growth")
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[1]
    for col, lab in [("loss", "loss"), ("policy_loss", "policy"), ("value_loss", "value")]:
        if col in df.columns:
            ax.plot(df["iteration"], df[col], marker="o", label=lab)
    ax.set_title("Training losses")
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[2]
    if "entropy" in df.columns:
        ax.plot(df["iteration"], df["entropy"], marker="o", label="policy entropy")
    if "value_mse" in df.columns:
        ax.plot(df["iteration"], df["value_mse"], marker="o", label="value mse")
    ax.set_title("Training diagnostics")
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[3]
    xs = np.asarray(benchmark_history.get("episode", []), dtype=float)
    gs = np.asarray(benchmark_history.get("global_score", []), dtype=float)
    gs_open = np.asarray(benchmark_history.get("global_score_open", []), dtype=float)
    if xs.size and gs.size:
        ax.plot(xs[:len(gs)], gs, marker="o", label="GS empty")
    if xs.size and gs_open.size:
        ax.plot(xs[:len(gs_open)], gs_open, marker="o", label="GS-open prefixes")
    ax.set_ylim(0.0, 1.02)
    ax.set_title("Project global scores")
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[4]
    by = benchmark_history.get("by_opponent", {})
    for label in ["Random", "Leftmost", "Center", "Lookahead-1", "Lookahead-3", "Lookahead-5", "Lookahead-7", "Lookahead-9", "Lookahead-11", "Lookahead-13"]:
        ys = by.get(label, [])
        if len(ys) > 0 and xs.size > 0:
            ax.plot(xs[:len(ys)], ys, marker=".", label=label.replace("Lookahead-", "L"))
    ax.set_ylim(0.0, 1.02)
    ax.set_title("Empty-board per-opponent win rates")
    ax.grid(True, alpha=0.3)
    ax.legend(ncols=4, fontsize=8)

    fig.suptitle(title, y=0.995, fontsize=14)
    fig.tight_layout()
    return fig

print("Plot helpers ready")

In [ ]:
# ============================================================
# 11. Initialize model / optimizer / replay
# ============================================================

if CFG.mode == "scratch":
    model = make_fresh_model(CFG)
    optimizer = make_optimizer(model, CFG)
    buffer = AZReplayBuffer(max_size=CFG.replay_max_size)
    history: List[Dict[str, Any]] = []
    benchmark_history = init_benchmark_history()
    start_iteration = 0
    print("Started fresh CNet192")

elif CFG.mode == "checkpoint":
    model, init_meta = load_flexible_cnet192_checkpoint(PROJECT_ROOT / CFG.init_checkpoint, CFG)
    optimizer = make_optimizer(model, CFG)
    buffer = AZReplayBuffer(max_size=CFG.replay_max_size)
    history = []
    benchmark_history = init_benchmark_history()
    start_iteration = 0
    print("Warm-started from:", CFG.init_checkpoint)

elif CFG.mode == "resume":
    model, optimizer, buffer, history, benchmark_history, start_iteration = load_az_checkpoint(PROJECT_ROOT / CFG.resume_checkpoint, CFG)

else:
    raise ValueError(f"Unknown CFG.mode={CFG.mode!r}")

model.to(DEVICE)
print(model.__class__.__name__)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Replay samples:", len(buffer))
print("Start iteration:", start_iteration)

In [ ]:
teacher_model = None

if bool(getattr(CFG, "use_teacher_anchor", False)):
    teacher_path = PROJECT_ROOT / str(CFG.teacher_checkpoint)
    teacher_model, teacher_meta = load_flexible_cnet192_checkpoint(teacher_path, CFG)
    teacher_model.to(DEVICE)
    teacher_model.eval()

    for p in teacher_model.parameters():
        p.requires_grad_(False)

    print("Teacher anchor loaded:", teacher_path)
    print("Teacher MCTS mix:", float(CFG.mcts_policy_mix))
    print("Teacher policy mix:", 1.0 - float(CFG.mcts_policy_mix))
else:
    print("Teacher anchor disabled")

In [ ]:
# ============================================================
# 12. Smoke test: one policy eval and one MCTS action on empty board
# ============================================================

board = empty_board()
pi_raw, v_raw = policy_value_for_board(model, board, player=1)
print("Raw policy probs:", np.round(pi_raw, 3), "value:", round(v_raw, 3), "greedy:", int(np.argmax(pi_raw)))

pi_mcts = run_mcts(
    model=model,
    board=board,
    player=1,
    cfg=CFG,
    rng=np.random.default_rng(CFG.seed),
    add_root_noise=False,
    visit_temperature=1.0,
)
print("MCTS policy:", np.round(pi_mcts, 3), "MCTS action:", int(np.argmax(pi_mcts)))

In [ ]:
plots_handle = display(HTML("<b>Live AlphaZero plot will appear here.</b>"), display_id=True)
fig = None

In [ ]:
# ============================================================
# 13. Main AlphaZero training loop
# ============================================================

end_iteration = int(start_iteration) + int(CFG.iterations_to_run)

for iteration in range(int(start_iteration) + 1, end_iteration + 1):
    print("\n" + "=" * 100)
    print(f"ALPHAZERO ITERATION {iteration}/{end_iteration} | run={CFG.run_name}")
    print("=" * 100)

    t0 = time.time()

    # --- self-play ---
    games = generate_self_play_batch(model, CFG, iteration=iteration)
    new_samples = 0
    w1 = w2 = d = 0
    plies = []
    for g in games:
        winner = int(g["winner"])
        w1 += int(winner == 1)
        w2 += int(winner == -1)
        d += int(winner == 0)
        plies.append(int(g["plies"]))
        for st, pi, z in g["samples"]:
            buffer.add(st, pi, z, augment_mirror=True)
            new_samples += 2

    # --- train ---
    train_stats = train_one_iteration(
        model,
        optimizer,
        buffer,
        CFG,
        iteration=iteration,
        teacher_model=teacher_model,
    )

    # --- history ---
    row = {
        "iteration": int(iteration),
        "new_games": int(len(games)),
        "samples": int(new_samples),
        "replay_size": int(len(buffer)),
        "winner_p1": int(w1),
        "winner_p2": int(w2),
        "draws": int(d),
        "avg_plies": float(np.mean(plies) if plies else np.nan),
        "elapsed_min": float((time.time() - t0) / 60.0),
        **train_stats,
    }
    history.append(row)
    print(pd.DataFrame([row]).T)

    # --- eval ---
    if CFG.eval_every > 0 and iteration % int(CFG.eval_every) == 0:
        eval_result = evaluate_project_hooks(model, CFG, iteration=iteration)
        update_benchmark_history(benchmark_history, iteration, eval_result)
        print("Eval summary:", {k: round(v, 4) for k, v in eval_result.items() if isinstance(v, float)})

    # --- save ---
    if CFG.save_every > 0 and iteration % int(CFG.save_every) == 0:
        save_az_checkpoint(model, optimizer, buffer, history, benchmark_history, CFG, iteration=iteration)
        export_cnet192_policy_checkpoint(
            model,
            RUN_DIR / f"{CFG.run_name}_cnet192_iter_{iteration:04d}.pt",
            CFG,
            iteration=iteration,
            replay_size=len(buffer),
        )

    # --- plot ---
    if CFG.plot_every > 0 and iteration % int(CFG.plot_every) == 0:
        fig = plot_live_training_az(history, benchmark_history, title=f"{CFG.run_name} AlphaZero")
        if fig is not None:
            fig.savefig(PLOT_DIR / f"{CFG.run_name}_live_iter_{iteration:04d}.png", dpi=140)
            plots_handle.update(fig)
            plt.close(fig)

In [ ]:
# ============================================================
# 14. Manual full evaluation cell
# Run this when a checkpoint looks interesting.
# ============================================================

FULL_TAG = f"{CFG.run_name}_manual_full"

suite_df, row_df = evaluate_and_log_to_excel(
    policy=model,
    opponents_cfg=EVALUATION_OPPONENTS,
    excel_path=str(PROJECT_ROOT / "Logs/AlphaZero/AZ_eval_FULL_empty.xlsx"),
    run_tag=FULL_TAG,
    device=DEVICE,
    lookahead=LA_SHARED,
    seed=CFG.seed + 12345,
    episodes=end_iteration if "end_iteration" in globals() else 0,
    progress=True,
    global_base=1.4,
)

display(suite_df)
display(row_df)

open_df = evaluate_opening_suite(
    model=model,
    opponents_cfg=EVALUATION_OPPONENTS,
    lookahead=LA_SHARED,
    seed=CFG.seed + 54321,
)
open_row = suite_to_summary_row(FULL_TAG, open_df, iteration=end_iteration if "end_iteration" in globals() else 0, score_col="GS_OPEN")
append_suite_row_to_excel(open_row, PROJECT_ROOT / "Logs/AlphaZero/AZ_eval_FULL_openings.xlsx")

display(open_df)
display(open_row)
print("GS:", float(row_df["GLOBAL_SCORE"].iloc[0]))
print("GS-open:", float(open_row["GS_OPEN"].iloc[0]))
print("GS_AVG:", 0.5 * (float(row_df["GLOBAL_SCORE"].iloc[0]) + float(open_row["GS_OPEN"].iloc[0])))

In [ ]:
# ============================================================
# 15. Optional: round-robin against selected checkpoints
# Fill MODEL_PATHS with CNet192-compatible checkpoints.
# ============================================================

MODEL_PATHS = {
    "PPO_926": "PPO_Models/PPO_926.pt",
    "PPO_1004": "PPO_Models/PPO_1004.pt",
    "AZ_latest": str(RUN_DIR / f"{CFG.run_name}_latest.pt"),  # AlphaZero ckpt needs manual extraction, see below
}

models = {}
for name, path in MODEL_PATHS.items():
    path = Path(path)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    if not path.exists():
        print("missing:", name, path)
        continue
    if "AlphaZero" in str(path) or path.name.endswith("_latest.pt"):
        ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
        m = make_fresh_model(CFG)
        m.load_state_dict(ckpt["model_state_dict"], strict=True)
    else:
        m, _ = load_flexible_cnet192_checkpoint(path, CFG)
    m.to(DEVICE).eval()
    models[name] = m

if len(models) >= 2:
    rr = round_robin_matrix(
        models,
        n_games=200,
        device=DEVICE,
        opening_noise_k=OPENING_NOISE_K,
        seed=CFG.seed,
        progress=True,
        paired_openings=True,
        opening_bias="center",
        with_meta_column=True,
        sort_by_meta=True,
    )
    display(rr)
    fig = plot_rr_heatmap(rr, title="AlphaZero / PPO round robin")
    plt.show()
else:
    print("Add at least two paths to MODEL_PATHS for round-robin.")

In [ ]:
end_time = time.time()
total_elapsed = (end_time - begin_start_time) / 60
print(f"A0 completed in {total_elapsed:.1f} minutes")